# ray-parametric-form — worked example 3: Solve for the parameter u where a ray crosses y=0

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `ray-parametric-form`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

Given a ray `R(u) = O + u*D`, the point where it crosses the horizontal plane `y = 0` is found by solving `O.y + u * D.y = 0`, giving `u = -O.y / D.y`. This algebraic inversion of the parametric form is the foundation of the ground-plane intersection test used in ray tracing. The result `u` can then be substituted back into `R(u)` to find the exact hit point.

## Worked solution

**Step 1 — Extract the y components.** `Oy = O[1]` (the y-coordinate of the origin) and `Dy = D[1]` (the y-component of the direction). These are scalars for a single ray.

**Step 2 — Solve for u.** Setting `Oy + u * Dy = 0` and solving: `u = -Oy / Dy`. This requires `Dy != 0` (ray must not be horizontal).

**Step 3 — Compute the hit point.** `H = O + u_hit * D`. The y-coordinate of `H` should be exactly 0 (or very close due to floating-point).

**Step 4 — Verify.** We check that `H[1]` (the y coordinate) is approximately zero and that `u_hit` is positive (the ray hits the plane in front, not behind).

In [ ]:
import torch as t

def ray_ground_intersection(ray: t.Tensor) -> tuple:
    """Find u and hit point H where ray crosses y=0. Requires D.y != 0."""
    O, D = ray[0], ray[1]
    # Solve: O.y + u * D.y = 0  =>  u = -O.y / D.y
    u_hit = -O[1] / D[1]
    H = O + u_hit * D
    return u_hit, H

t.manual_seed(63)

# Ray starting above y=0, pointing downward
O = t.tensor([3.0, 5.0, 1.0])     # above the ground (y=5)
D = t.tensor([0.5, -1.0, 0.2])    # pointing down (Dy = -1)
ray = t.stack([O, D])

u_hit, H = ray_ground_intersection(ray)
print(f'u_hit = {u_hit.item():.4f}')       # 5.0 (5 / 1)
print(f'Hit point: {H.tolist()}')           # y should be ~0
print(f'H.y ≈ 0: {abs(H[1].item()) < 1e-5}')  # True
print(f'u_hit > 0: {u_hit.item() > 0}')    # True (hit is in front)